## Search Algorithms: Uniform Cost Search and A* Search

These algorithms are used to find the shortest path from a starting node to a goal node in a graph or tree structure.

### Node Class
First, let's define a `Node` class that will represent a state in our search space. Each node will store its state, its parent node (to reconstruct the path), the cost to reach it from the start node, and for A* search, an estimated heuristic cost to the goal.

In [ ]:
import heapq

class Node:
    def __init__(self, state, parent=None, cost=0, heuristic=0):
        self.state = state
        self.parent = parent
        self.cost = cost
        self.heuristic = heuristic

    def __lt__(self, other):
        # This is important for heapq to compare Nodes based on their total cost (f_cost)
        return (self.cost + self.heuristic) < (other.cost + other.heuristic)

    def __eq__(self, other):
        return self.state == other.state

    def __hash__(self):
        return hash(self.state)

    def __repr__(self):
        return f"Node(state={self.state}, cost={self.cost}, heuristic={self.heuristic})"

### Uniform Cost Search (UCS)

Uniform Cost Search is an uninformed search algorithm that finds the path with the lowest cumulative cost from the start node to a goal node. It prioritizes expanding nodes with the lowest path cost (`g(n)`).

It uses a priority queue to always expand the node with the lowest path cost. It guarantees finding the optimal path in graphs with non-negative edge weights.

In [ ]:
def uniform_cost_search(graph, start_state, goal_state):
    start_node = Node(start_state, cost=0)
    priority_queue = [(0, start_node)]  # (cost, node)
    visited = {start_state: start_node}

    while priority_queue:
        current_cost, current_node = heapq.heappop(priority_queue)

        if current_node.state == goal_state:
            path = []
            while current_node:
                path.append(current_node.state)
                current_node = current_node.parent
            return path[::-1] # Return reversed path

        for neighbor_state, edge_cost in graph.get(current_node.state, {}).items():
            new_cost = current_node.cost + edge_cost
            if neighbor_state not in visited or new_cost < visited[neighbor_state].cost:
                neighbor_node = Node(neighbor_state, parent=current_node, cost=new_cost)
                visited[neighbor_state] = neighbor_node
                heapq.heappush(priority_queue, (new_cost, neighbor_node))
    return None # No path found

### A* Search Algorithm

A* search is an informed search algorithm that uses both the cost to reach a node (`g(n)`) and a heuristic estimate of the cost from that node to the goal (`h(n)`). It prioritizes nodes with the lowest total estimated cost (`f(n) = g(n) + h(n)`).

A* guarantees an optimal path if the heuristic function is admissible (never overestimates the cost to the goal) and consistent (for every node `n` and every successor `n'` of `n`, the estimated cost `h(n)` is no more than the cost to get from `n` to `n'` plus the estimated cost `h(n')`).

In [ ]:
def a_star_search(graph, start_state, goal_state, heuristic_function):
    start_node = Node(start_state, cost=0, heuristic=heuristic_function(start_state, goal_state))
    priority_queue = [(start_node.cost + start_node.heuristic, start_node)] # (f_cost, node)
    visited = {start_state: start_node}

    while priority_queue:
        f_cost, current_node = heapq.heappop(priority_queue)

        if current_node.state == goal_state:
            path = []
            while current_node:
                path.append(current_node.state)
                current_node = current_node.parent
            return path[::-1]

        for neighbor_state, edge_cost in graph.get(current_node.state, {}).items():
            new_cost = current_node.cost + edge_cost
            # Check if this new path to neighbor is better or if neighbor hasn't been visited yet
            if neighbor_state not in visited or new_cost < visited[neighbor_state].cost:
                neighbor_node = Node(neighbor_state, parent=current_node, cost=new_cost,
                                     heuristic=heuristic_function(neighbor_state, goal_state))
                visited[neighbor_state] = neighbor_node
                heapq.heappush(priority_queue, (neighbor_node.cost + neighbor_node.heuristic, neighbor_node))
    return None # No path found

### Example Usage

Let's consider a simple graph and apply both UCS and A* search.

In [ ]:
# Define a simple graph
graph = {
    'A': {'B': 1, 'C': 4},
    'B': {'D': 2, 'E': 5},
    'C': {'F': 1},
    'D': {'G': 3},
    'E': {'G': 1},
    'F': {'G': 5},
    'G': {}
}







heuristic_values = {
    'A': 6, 'B': 4, 'C': 2,
    'D': 2, 'E': 1, 'F': 1,
    'G': 0
}

def heuristic(state, goal_state):
    return heuristic_values.get(state, 0)

start = 'A'
goal = 'G'

# Run Uniform Cost Search
ucs_path = uniform_cost_search(graph, start, goal)
print(f"Uniform Cost Search Path: {ucs_path}")

# Run A* Search
a_star_path = a_star_search(graph, start, goal, heuristic)
print(f"A* Search Path: {a_star_path}")